# 2023年Java大厂面试｜10：企业场景篇

> 学习目标：把设计模式、统一认证、权限控制、数据安全、日志检索和生产排障串成可落地的面试答案。

**资料范围**：主要依据网盘 `5.2023版Java面试教程/10-企业场景篇/PPT` 中的 `10-设计模式.pptx`（28页）和 `11-常见技术场景.pptx`（27页）整理，并补充必要的工程化答题框架。


## 目录与答题主线

| 模块 | 必会问题 | 记忆主线 |
|---|---|---|
| 设计模式 | 工厂、策略、责任链怎么选 | 解耦创建、替换算法、串联处理 |
| 认证与权限 | SSO、JWT、RBAC 如何落地 | 身份认证、令牌传递、授权判断 |
| 数据安全 | 上传文件和敏感数据如何保护 | 校验、隔离、加密、审计 |
| 日志平台 | ELK 如何采集和检索 | 采集、存储搜索、可视化 |
| 生产排障 | 慢请求、异常、CPU 高怎么查 | 先证据、再定位、后修复 |
| 性能保障 | 压测、监控、链路追踪、Arthas | 指标、瓶颈、验证闭环 |


## 一、设计模式在企业项目中的价值

设计模式不是为了把代码写得复杂，而是把经常变化的部分隔离出来。面试回答可以按三步展开：先说业务变化点，再说模式如何降低耦合，最后说明实际收益和代价。

- **工厂类模式**：隐藏对象创建细节，调用方依赖抽象；
- **策略模式**：把可替换的业务算法封装成多个策略；
- **责任链模式**：把多个处理节点串起来，请求沿链路传递。

选择模式时要结合规模。只有两个简单分支时不必为了套模式增加层次；当登录方式、支付方式、审核节点持续增加时，模式才会带来清晰收益。


## 二、简单工厂模式

简单工厂由一个工厂类根据参数创建不同产品，调用方只关心产品接口。例如：`LoginService login = LoginFactory.create(type);`。

```java
interface LoginService { User login(String credential); }

class LoginFactory {
    static LoginService create(String type) {
        return switch (type) {
            case "password" -> new PasswordLogin();
            case "sms" -> new SmsLogin();
            default -> throw new IllegalArgumentException("unsupported login type");
        };
    }
}
```

优点是使用简单、集中创建逻辑；缺点是每增加一种产品都要修改工厂，容易违反开闭原则，分支多时工厂会变成“大类”。


## 三、工厂方法模式

工厂方法把“创建哪一种产品”延迟到具体工厂。抽象工厂只定义创建方法，`PasswordLoginFactory`、`SmsLoginFactory` 分别创建对应产品。

```java
interface LoginFactory { LoginService create(); }
final class SmsLoginFactory implements LoginFactory {
    public LoginService create() { return new SmsLogin(); }
}
```

新增产品时增加产品类和工厂类即可，已有代码改动小，更符合开闭原则。代价是类的数量会增加，通常需要结合依赖注入或注册表管理工厂。


## 四、抽象工厂模式

抽象工厂用于创建一组相互匹配的产品，即一个“产品族”。例如同一套主题需要同时创建按钮、输入框和菜单，Windows 工厂与 Mac 工厂分别提供一整套实现。

它适合“同一族产品需要保持一致”的场景，不适合只有一个产品维度的简单创建。增加新的产品等级会修改所有具体工厂，因此要先确认变化方向：抽象工厂更容易扩展产品族，不容易扩展产品等级。


## 五、策略模式：封装可替换算法

策略模式把一组可互换的算法封装为统一接口，运行时选择其中一个。例如支付、优惠计算、物流计费、Excel 解析、登录方式都可以使用策略。

```java
import java.math.BigDecimal;

interface DiscountStrategy {
    BigDecimal finalPrice(BigDecimal originalPrice);
}
class VipDiscount implements DiscountStrategy {
    public BigDecimal finalPrice(BigDecimal originalPrice) {
        return originalPrice.multiply(new BigDecimal("0.90"));
    }
}
class PriceService {
    private final DiscountStrategy strategy;
    PriceService(DiscountStrategy strategy) { this.strategy = strategy; }
    BigDecimal finalPrice(BigDecimal originalPrice) {
        return strategy.finalPrice(originalPrice);
    }
}
```

示例只表达策略结构；实际价格计算还需约定币种、精度、舍入方式和优惠叠加顺序。

策略的核心收益是把变化的业务规则与稳定的流程分开，避免控制器或服务类堆积 `if-else`。策略选择本身也可以交给工厂、Spring 容器或按类型注册的 Map。


## 六、工厂 + 策略实现多种登录

登录流程通常稳定：接收请求、校验参数、记录审计、生成登录结果；变化的是密码、短信、微信、企业微信等认证算法。可以让工厂负责选择策略，让策略负责具体认证。

```java
class LoginStrategyFactory {
    private final Map<String, LoginStrategy> strategies;
    LoginStrategy get(String type) {
        LoginStrategy strategy = strategies.get(type);
        if (strategy == null) throw new IllegalArgumentException("unsupported type");
        return strategy;
    }
}
```

面试时补充三个工程细节：策略类型来自白名单；每个策略有独立超时和失败处理；认证成功后统一由登录服务签发会话或 JWT，避免不同策略各自实现令牌逻辑。


## 七、责任链模式

责任链把多个处理器按顺序连接，每个处理器决定“自己处理、拒绝请求，还是交给下一个节点”。典型结构是 `Handler` 持有 `next` 引用。

订单处理可以拆成库存校验、优惠校验、风控校验、支付校验和落库；内容审核可以拆成敏感词、图片、广告和人工复核节点；审批流程也可以按角色串联。

责任链的优点是节点可插拔、顺序清晰；风险是链路过长时调试困难、节点顺序会影响结果。实际项目应记录节点名称、耗时、拒绝原因，并为链路设置超时和兜底处理。


## 八、模式选择的面试模板

遇到“项目中为什么这样设计”，可以这样回答：

1. 先指出变化点，例如登录方式和优惠规则会持续增加；
2. 说明抽象边界，例如稳定流程依赖接口，变化规则放到实现类；
3. 说明模式组合，例如工厂选择策略，责任链串联审核节点；
4. 说清代价，例如类数量增加、需要注册配置和链路监控；
5. 最后给出验证方式，例如新增一种策略只增加实现类和配置，不修改核心流程。


## 九、单点登录（SSO）

SSO（Single Sign On）让用户登录一次，就能访问多个相互信任的业务系统。核心角色通常包括统一认证中心、业务系统、浏览器和网关。

基本流程：用户访问业务系统；业务系统发现没有本地会话，跳转认证中心；认证中心完成登录后返回一次性凭证或授权码；业务系统向认证中心换取用户身份，建立自己的会话。退出登录时还要考虑各系统会话同步失效。

设计时要明确信任边界、回调地址白名单、票据有效期、重放防护和跨域策略。不要把密码直接交给每个业务系统，密码校验应集中在认证中心。

**SSO 与 JWT 不同层次**：SSO 是统一登录能力，JWT 是令牌格式。使用 JWT 不等于自动实现单点登录，也不等于实现统一退出；跨域身份跳转、会话建立和退出传播仍要设计。


## 十、JWT 网关认证流程

JWT 常用于无状态的访问令牌。典型流程如下：

1. 用户提交账号密码，登录服务完成认证；
2. 服务签发包含用户标识、角色或权限摘要、过期时间的 JWT；
3. 浏览器保存令牌，后续请求在 `Authorization: Bearer <token>` 中携带；
4. 网关校验签名、过期时间和必要声明；
5. 校验失败返回 `401`，通过后把用户信息传给下游服务。

JWT 适合快速校验，但令牌一旦签发，在过期前可能仍然有效。需要提前失效时可以使用短有效期 + 刷新令牌、黑名单或版本号。不要把密码、身份证号等敏感信息放进 JWT；常见签名 JWT 的载荷只是编码，拿到令牌的人可以读到内容。签名校验、密钥轮换、算法白名单以及签发者和受众检查必须统一管理。

浏览器令牌保存方式要结合 XSS 和 CSRF 威胁设计：脚本可读的存储有令牌窃取风险，Cookie 应结合 HttpOnly、Secure、SameSite 和必要的 CSRF 防护。网关向下游传递身份时，要清理客户端伪造的身份头，并保护内部服务入口。


## 十一、认证、授权与会话的边界

- **认证（Authentication）**：你是谁，例如校验密码、短信验证码或第三方身份；
- **授权（Authorization）**：你能做什么，例如判断是否有订单导出权限；
- **会话（Session）**：如何在连续请求中识别你，例如 Cookie、Session 或 JWT。

网关可以完成令牌的基础校验，但细粒度权限仍应由业务服务结合资源归属判断。例如“能否查看订单”不仅取决于角色，还取决于订单是否属于当前用户。


## 十二、RBAC 权限模型

RBAC（Role-Based Access Control）以角色为中间层：用户拥有角色，角色拥有权限。基础模型通常包含五张关系表：用户表、角色表、权限表、用户角色关系表、角色权限关系表。若系统需要菜单、目录或数据范围，可扩展用户、角色、菜单和关联关系。

权限通常拆成资源和操作，例如 `order:read`、`order:export`。登录后可以把角色或权限标识放入会话上下文，但每次访问敏感资源仍要在服务端校验，不能只依赖前端隐藏按钮。

RBAC 的优点是授权关系复用，缺点是角色爆炸和数据权限表达复杂。遇到组织、部门、本人数据范围时，可在 RBAC 上增加数据权限规则或 ABAC 条件。


## 十三、Spring Security / Shiro 的落地思路

Spring Security 常通过过滤器链完成认证、授权、CSRF、会话和异常处理；Shiro 也提供认证、授权、会话和加密能力。面试时不必只背框架名，要说明请求经过哪些边界：

- 登录接口校验凭证并建立认证上下文；
- 过滤器或拦截器解析令牌；
- 授权注解、表达式或权限服务判断操作；
- 未认证返回 `401`，已认证但无权限返回 `403`；
- 统一记录登录失败、越权和敏感操作审计日志。

权限失败的状态码要区分：`401` 表示身份凭证缺失或无效，`403` 表示身份已确认但不允许访问。


## 十四、上传数据安全

文件上传不能只看扩展名。完整校验至少包括：文件大小上限、数量上限、文件名清理、扩展名白名单、真实 MIME 类型和文件头校验。上传文件应保存到应用目录之外或对象存储中，使用随机文件名，避免路径穿越和覆盖已有文件。

对图片、压缩包、Office 文档等格式，还要考虑恶意脚本、压缩炸弹、宏和病毒扫描。下载时通过受控接口返回，设置正确的 `Content-Type` 和 `Content-Disposition`，必要时使用短期签名 URL。所有失败原因和操作者应进入审计日志。


## 十五、对称加密与非对称加密

**对称加密**使用同一把密钥加密和解密，速度快，适合大量数据或文件内容；难点是密钥分发、轮换和安全保存。AES 是常见选择，实际使用还要正确设置随机 IV、工作模式和认证标签。

**非对称加密**使用公钥和私钥，便于身份验证和密钥交换，但计算开销较大，适合少量数据、签名和协商会话密钥。RSA、ECC 属于常见体系。

工程上常采用混合加密：用非对称方式安全交换临时对称密钥，再用对称方式加密大数据；用数字签名验证来源和完整性。密码不能用可逆加密保存，应使用带盐的慢哈希算法。


## 十六、ELK 日志系统

ELK 由三部分组成：

- **Elasticsearch**：分布式存储、倒排索引、检索和聚合分析；
- **Logstash**：采集、过滤、解析和转发日志；
- **Kibana**：查询、可视化、仪表盘和告警。

生产系统常把应用日志输出为 JSON，包含时间、级别、服务名、环境、线程、traceId、用户标识和异常堆栈。Logstash 或轻量采集器负责补充字段并发送到 Elasticsearch，Kibana 按服务、时间和 traceId 检索。

日志平台要控制索引生命周期、分片数量、字段基数、敏感信息脱敏和访问权限，避免把密码、令牌和完整身份证号写进日志。


## 十七、Linux 日志查看命令

```bash
tail -f app.log                         # 持续查看新增日志
tail -n 100 -f app.log                  # 从最后 100 行开始跟踪
head -n 100 app.log                     # 查看开头 100 行
cat -n app.log | grep "ERROR"          # 带行号检索关键词
cat -n app.log | grep "ERROR" > err.txt
sed -n '100,150p' app.log                # 查看第 100～150 行
grep -n -C 10 'request-id' app.log       # 关键词及前后 10 行
```

排查时先按时间和 requestId 缩小范围，再结合上下文查看异常堆栈。线上文件可能正在滚动或被压缩，需确认当前日志文件、时区、编码和权限；不要在高流量日志上反复执行代价很大的全盘搜索。


## 十八、生产问题排查方法

遇到接口慢、错误率上升或服务不可用，先建立时间线：故障从什么时候开始、影响哪些接口、是否伴随发布或流量变化。随后按应用、数据库、缓存、消息队列和网络边界逐层排查。

优先查看业务日志和系统日志，确认异常类型、请求参数摘要、traceId 和下游耗时；再对照监控中的 QPS、响应时间、错误率、CPU、内存、磁盘 IO、连接池和线程池。每次改动都记录假设、操作和结果，避免“调参式排障”。


## 十九、远程 Debug 的使用边界

测试环境可通过 JDWP 连接远程 JVM，例如：

```bash
java -jar -agentlib:jdwp=transport=dt_socket,server=y,suspend=n,address=5005 project-1.0-SNAPSHOT.jar
```

远程 Debug 会增加暂停线程、泄露调试端口和影响吞吐的风险。必须限制网络访问、使用与线上一致的代码版本，优先在测试或灰度环境复现；生产问题通常先使用日志、线程转储、指标和低风险诊断工具。断点命中时要关注锁、连接和超时，避免长时间挂起请求。


## 二十、压测指标与容量判断

压测至少关注：响应时间（平均值、P95/P99）、QPS、并发数、吞吐量、CPU、内存、磁盘 IO、网络带宽、连接池、线程池、GC 和错误率。只看平均响应时间会掩盖尾延迟，容量判断应结合 P99 和错误率。

压测前明确场景、数据规模、目标 SLA、阶梯加压方式和停止条件；压测环境与线上配置尽量一致。发现瓶颈后先确认是 CPU、锁竞争、慢 SQL、外部依赖、连接池还是 GC，再做单变量优化并复测。

QPS 是单位时间完成的请求数，并发数是同时处理或等待的请求数。在稳定系统中，可以用“平均在途请求数 ≈ 吞吐量 × 平均响应时间”帮助理解两者关系；线程数、虚拟用户数与服务端在途请求数不能直接等同。


## 二十一、监控、链路追踪与 Arthas

- **Prometheus + Grafana**：采集和展示时序指标，适合 QPS、延迟、资源和业务计数器；
- **SkyWalking / Zipkin**：通过 traceId 连接跨服务调用，定位慢链路和异常下游；
- **Arthas**：在线诊断类、方法调用、线程和热点问题，常用 `dashboard`、`thread`、`jvm`、`memory`、`watch`、`trace` 等命令。

诊断工具要设置权限、采样范围和执行时长，避免在高峰期输出过多数据。链路追踪应在入口生成 traceId，并在日志、消息和下游请求中透传。


## 二十二、综合场景：一次订单异常如何回答

如果面试官问“订单接口偶发超时怎么处理”，可以按以下路径回答：先在网关和业务日志中用 traceId 找到慢请求；查看接口 P99、线程池、数据库连接池和下游调用耗时；用链路追踪确定瓶颈位置；必要时在测试或灰度环境用 Arthas 查看热点方法和线程状态；通过压测复现后，再针对慢 SQL、锁竞争、外部依赖超时或 GC 做修复。

如果涉及权限，则先在网关校验 JWT，再由订单服务做 RBAC 和资源归属判断；如果涉及文件，则校验类型与大小并保存到受控对象存储；所有关键操作写入结构化审计日志。这个回答能把模式、认证、安全、日志和排障串在同一个业务场景里。


## 二十三、常见面试速答

- **简单工厂和工厂方法的区别？** 简单工厂集中分支创建对象，使用简单但新增类型要改工厂；工厂方法把创建职责分给具体工厂，更符合开闭原则。
- **为什么策略模式能减少 if-else？** 每种算法独立实现，主流程只依赖策略接口，选择逻辑可由工厂或注册表集中管理。
- **JWT 一定安全吗？** 不一定。必须校验签名、过期时间和算法，保护密钥，控制载荷，配合短期令牌和刷新/失效机制。
- **401 和 403 的区别？** 401 是未认证或凭证无效，403 是已认证但没有权限。
- **RBAC 的三要素是什么？** 用户、角色、权限；用户通过角色获得权限。
- **ELK 各组件做什么？** Logstash 采集处理，Elasticsearch 存储检索分析，Kibana 查询和可视化。
- **生产问题怎么排查？** 先保留日志和指标证据，再定位边界和根因，最后在可控环境验证修复。
- **压测看哪些指标？** P95/P99、QPS、并发、错误率，以及 CPU、内存、IO、连接池、线程池和 GC。


## 二十四、自测清单

1. 什么时候用简单工厂，什么时候升级为工厂方法？
2. 抽象工厂适合扩展产品族还是产品等级？
3. 登录、支付、优惠和物流计费为什么适合策略模式？
4. 责任链节点如何记录拒绝原因和耗时？
5. SSO 中认证中心、业务系统和浏览器如何交互？
6. JWT 网关校验通过后为什么仍要做业务授权？
7. RBAC 五张基础表分别是什么？如何处理数据权限？
8. 上传文件如何防止路径穿越、伪造类型和压缩炸弹？
9. 对称加密和非对称加密如何组合使用？
10. ELK 中如何用 traceId 串起一次请求？
11. 如何从日志、监控和链路追踪定位接口超时？
12. 远程 Debug 为什么要限制在测试或灰度环境？
13. 为什么压测不能只看平均响应时间？


## 资料说明与版本提示

- `10-设计模式.pptx` 共 28 页，重点覆盖简单工厂、工厂方法、抽象工厂、策略、工厂 + 策略和责任链；
- `11-常见技术场景.pptx` 共 27 页，重点覆盖 SSO、JWT、RBAC、Spring Security / Shiro、上传安全、加密、ELK、Linux 日志、远程 Debug、压测、监控、链路追踪和 Arthas；
- 本篇按面试复习重新组织了课件内容，示例代码用于说明结构，未作为完整课程工程运行；
- 未观看视频、未运行课程项目、未进行真实压测或线上验证；具体参数、组件版本和安全策略仍需结合目标项目确认。
